In [ ]:

import pandas as pd
import numpy as np


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving expenses.csv to expenses (1).csv


In [ ]:

df = pd.read_csv("expenses.csv")  #  reading the file
print(f" Loaded {len(df)} rows, {len(df.columns)} columns")
print("\nRaw Data Preview:")
print(df.head(10).to_string(index=False))

 Loaded 24 rows, 7 columns

Raw Data Preview:
 expense_id  user_id     user_name      category  amount expense_date          description
          1        1 Alice Johnson          Food $150.00   2025-04-02     Grocery shopping
          2        1 Alice Johnson     Transport  $45.00   2025-04-05     Monthly bus pass
          3        1 Alice Johnson     Utilities $120.00   2025-04-10     Electricity bill
          4        1 Alice Johnson Entertainment  $30.00   2025-04-15 Netflix subscription
          5        1 Alice Johnson          Food  $80.00   2025-04-20    Restaurant dinner
          6        1 Alice Johnson        Health  $75.00   2025-04-28       Gym membership
          7        2     Bob Smith     Transport  $60.00   2025-04-03            Cab rides
          8        2     Bob Smith        Health $200.00   2025-04-08         Doctor visit
          9        2     Bob Smith          Food $110.00   2025-04-12     Weekly groceries
         10        2     Bob Smith Entertain

In [ ]:

df["amount"] = (
    df["amount"]
    .astype(str)
    .str.replace(r"[\$,\s]", "", regex=True)
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")


bad_amount = df[df["amount"].isna()]
if not bad_amount.empty:
    print(f"\n  Found {len(bad_amount)} rows with invalid amounts:")
    print(bad_amount[["expense_id", "user_name", "category", "description"]].to_string(index=False))

df.dropna(subset=["amount"], inplace=True)  #dropping invalid data
print(f"\n Rows after dropping invalid amounts: {len(df)}")


df["expense_date"] = pd.to_datetime(df["expense_date"], format="mixed", dayfirst=False, errors="coerce")

bad_dates = df[df["expense_date"].isna()] #date format to accept all types of dates
if not bad_dates.empty:
    print(f"\n  Found {len(bad_dates)} rows with unparseable dates — dropping.")
    print(bad_dates[["expense_id", "description"]].to_string(index=False))

df.dropna(subset=["expense_date"], inplace=True)
print(f" Rows after dropping invalid dates: {len(df)}")


df["month"] = df["expense_date"].dt.to_period("M")


df["category"]  = df["category"].str.strip().str.title()
df["user_name"] = df["user_name"].str.strip().str.title()

df.reset_index(drop=True, inplace=True)

print("\n Cleaned Data Sample:")
print(df[["expense_id", "user_name", "category", "amount", "expense_date", "month"]].head(10).to_string(index=False))




  Found 2 rows with invalid amounts:
 expense_id     user_name      category          description
         22 Alice Johnson Entertainment Missing amount entry
         23     Bob Smith          Food     Corrupted amount

 Rows after dropping invalid amounts: 22
 Rows after dropping invalid dates: 22

 Cleaned Data Sample:
 expense_id     user_name      category  amount expense_date   month
          1 Alice Johnson          Food   150.0   2025-04-02 2025-04
          2 Alice Johnson     Transport    45.0   2025-04-05 2025-04
          3 Alice Johnson     Utilities   120.0   2025-04-10 2025-04
          4 Alice Johnson Entertainment    30.0   2025-04-15 2025-04
          5 Alice Johnson          Food    80.0   2025-04-20 2025-04
          6 Alice Johnson        Health    75.0   2025-04-28 2025-04
          7     Bob Smith     Transport    60.0   2025-04-03 2025-04
          8     Bob Smith        Health   200.0   2025-04-08 2025-04
          9     Bob Smith          Food   110.0   2025

In [ ]:

amounts = df["amount"].to_numpy()

print(f"\n Overall Statistics (NumPy):")
print(f"   Total Expenses  : ₹{np.sum(amounts):>10.2f}")
print(f"   Mean per Entry  : ₹{np.mean(amounts):>10.2f}")
print(f"   Median          : ₹{np.median(amounts):>10.2f}")
print(f"   Std Deviation   : ₹{np.std(amounts):>10.2f}")
print(f"   Min Expense     : ₹{np.min(amounts):>10.2f}")
print(f"   Max Expense     : ₹{np.max(amounts):>10.2f}")


print("\n Monthly Totals per User (NumPy):")
for user in df["user_name"].unique():
    user_df = df[df["user_name"] == user]
    months  = user_df["month"].unique()
    print(f"\n   {user}")
    for month in sorted(months):
        mask   = user_df["month"] == month
        total  = np.sum(user_df.loc[mask, "amount"].to_numpy())
        avg    = np.mean(user_df.loc[mask, "amount"].to_numpy())
        print(f"     {month}  →  Total: ₹{total:>8.2f}  |  Avg per transaction: ₹{avg:>7.2f}")



 Overall Statistics (NumPy):
   Total Expenses  : ₹   1980.50
   Mean per Entry  : ₹     90.02
   Median          : ₹     82.50
   Std Deviation   : ₹     52.16
   Min Expense     : ₹     15.00
   Max Expense     : ₹    200.00

 Monthly Totals per User (NumPy):

   Alice Johnson
     2025-02  →  Total: ₹  330.00  |  Avg per transaction: ₹ 165.00
     2025-03  →  Total: ₹  325.00  |  Avg per transaction: ₹ 108.33
     2025-04  →  Total: ₹  555.50  |  Avg per transaction: ₹  79.36

   Bob Smith
     2025-02  →  Total: ₹  125.00  |  Avg per transaction: ₹  62.50
     2025-03  →  Total: ₹  170.00  |  Avg per transaction: ₹  56.67
     2025-04  →  Total: ₹  475.00  |  Avg per transaction: ₹  95.00


In [ ]:


monthly_expense = (
    df.groupby(["user_name", "month", "category"])["amount"]
    .sum()
    .unstack(fill_value=0)
)

print("\n Monthly Expense Breakdown by Category:")
print(monthly_expense.to_string())



 Monthly Expense Breakdown by Category:
category               Entertainment   Food  Health  Transport  Utilities
user_name     month                                                      
Alice Johnson 2025-02            0.0  200.0     0.0        0.0      130.0
              2025-03            0.0  160.0     0.0       50.0      115.0
              2025-04           30.0  285.5    75.0       45.0      120.0
Bob Smith     2025-02            0.0   85.0     0.0       40.0        0.0
              2025-03           25.0   95.0    50.0        0.0        0.0
              2025-04           15.0  110.0   200.0       60.0       90.0


In [ ]:


monthly_summary = (
    df.groupby(["user_name", "month"])["amount"]
    .agg(
        total_spent="sum",
        num_transactions="count",
        avg_transaction="mean",
        max_single_expense="max"
    )
    .round(2)
    .reset_index()
)

print(monthly_summary.to_string(index=False))



STEP 5: Per-User Monthly Summary
    user_name   month  total_spent  num_transactions  avg_transaction  max_single_expense
Alice Johnson 2025-02        330.0                 2           165.00               200.0
Alice Johnson 2025-03        325.0                 3           108.33               160.0
Alice Johnson 2025-04        555.5                 7            79.36               150.0
    Bob Smith 2025-02        125.0                 2            62.50                85.0
    Bob Smith 2025-03        170.0                 3            56.67                95.0
    Bob Smith 2025-04        475.0                 5            95.00               200.0


In [ ]:


top_category = (
    df.groupby(["user_name", "month", "category"])["amount"]
    .sum()
    .reset_index()
    .sort_values(["user_name", "month", "amount"], ascending=[True, True, False])
    .groupby(["user_name", "month"])
    .first()
    .reset_index()
    .rename(columns={"category": "top_category", "amount": "top_category_spend"})
)

print(top_category.to_string(index=False))


    user_name   month top_category  top_category_spend
Alice Johnson 2025-02         Food               200.0
Alice Johnson 2025-03         Food               160.0
Alice Johnson 2025-04         Food               285.5
    Bob Smith 2025-02         Food                85.0
    Bob Smith 2025-03         Food                95.0
    Bob Smith 2025-04       Health               200.0


In [ ]:


anomalies_list = []

for user in df["user_name"].unique():
    user_df = df[df["user_name"] == user].copy()
    mean    = np.mean(user_df["amount"].to_numpy())
    std     = np.std(user_df["amount"].to_numpy())
    threshold = mean + 1.5 * std

    flagged = user_df[user_df["amount"] > threshold].copy()
    flagged["user_mean"]  = round(mean, 2)
    flagged["threshold"]  = round(threshold, 2)
    anomalies_list.append(flagged)

anomalies = pd.concat(anomalies_list)

if anomalies.empty:
    print(" No anomalies detected.")
else:
    print(f" {len(anomalies)} unusual expense(s) flagged:\n")
    print(
        anomalies[["user_name", "category", "amount", "expense_date", "description", "user_mean", "threshold"]]
        .to_string(index=False)
    )


 2 unusual expense(s) flagged:

    user_name category  amount expense_date  description  user_mean  threshold
Alice Johnson     Food   200.0   2025-02-10 Bulk grocery     100.88     177.17
    Bob Smith   Health   200.0   2025-04-08 Doctor visit      77.00     153.06


In [ ]:




df.to_csv("expenses_cleaned.csv", index=False)
print(" Cleaned data   → expenses_cleaned.csv")


monthly_summary.to_csv("monthly_summary.csv", index=False)
print("Monthly summary → monthly_summary.csv")


category_breakdown = (
    df.groupby(["user_name", "month", "category"])["amount"]
    .sum()
    .reset_index()
    .rename(columns={"amount": "total_spent"})
)
category_breakdown.to_csv("category_breakdown.csv", index=False)
print(" Category breakdown → category_breakdown.csv")


if not anomalies.empty:
    anomalies.to_csv("anomalies.csv", index=False)
    print(" Anomalies       → anomalies.csv")





STEP 8: Exporting Results
 Cleaned data   → expenses_cleaned.csv
Monthly summary → monthly_summary.csv
 Category breakdown → category_breakdown.csv
 Anomalies       → anomalies.csv
